## Part A: A small ratings matrix

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

movies = ["Action 1", "Action 2", "Action 3",
          "Romance 1", "Romance 2", "Romance 3"]
users = ["Ana", "Ben", "Cal", "Dee", "Eve", "Fay"]
#list sample movies and users

In [ ]:
toy = np.array([
    [5, 4, 5, 1, np.nan, 2],
    [4, np.nan, 5, 2, 1, 1],
    [5, 5, np.nan, 1, 2, np.nan],
    [1, 2, 1, 5, 4, np.nan],
    [np.nan, 1, 2, 4, 5, 5],
    [2, 1, np.nan, 5, np.nan, 4],
])

toy_table = pd.DataFrame(toy, index=users, columns=movies)
toy_table
#build ratings table with missing values

In [ ]:
missing = np.isnan(toy).sum()
print("Missing entries:", missing, "out of", toy.size)
#count missing ratings in sample data

## Part B: The MovieLens data

In [ ]:
import os
import urllib.request
import zipfile

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip"
    )
    zipfile.ZipFile("ml-100k.zip").extractall()

In [ ]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"]
)
ratings.head()
#load rating files into a table

In [ ]:
print("Number of ratings:", len(ratings))
print("Number of users:", ratings["user"].nunique())
print("Number of movies:", ratings["item"].nunique())
#display ratings, users, and movies

In [ ]:
star_counts = ratings["rating"].value_counts().sort_index()
star_counts
#count the number of ratings for each star score

In [ ]:
average_rating = ratings["rating"].mean()
print("Average rating:", round(average_rating, 3))
#calculate average rating score

In [ ]:
star_counts.plot(kind="bar", color="steelblue")
plt.xlabel("Stars")
plt.ylabel("Number of ratings")
plt.title("MovieLens 100K: How people rate")
plt.show()
#plot counts for each star rating

## Part C: Building the user-by-movie matrix

In [ ]:
R = ratings.pivot_table(
    index="user",
    columns="item",
    values="rating"
)
#build user-movie ratings matrix

print("Matrix shape:", R.shape)

fraction_missing = R.isna().to_numpy().mean()
print("Fraction missing:", round(fraction_missing, 4))
#calculate fraction of missing ratings

per_user = ratings.groupby("user").size()

print("Least active user:", per_user.min())
print("Median user:", int(per_user.median()))
print("Most active user:", per_user.max())

## Part D: Which movies are rated most?

In [ ]:
titles = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item", "title"]
)

stats = ratings.groupby("item")["rating"].agg(["count", "mean"])
stats = stats.join(titles.set_index("item"))

top_five = stats.sort_values("count", ascending=False).head(5)
top_five.round(2)
#display the top five most rated movies and their average ratings